In [0]:
STORAGE_ACCOUNT = 'stccasemauricioes'
CONTAINER = 'data'
STORAGE_KEY = 'COLOCAR_CHAVE_AQUI'

spark.conf.set("fs.azure.account.key." + STORAGE_ACCOUNT + ".blob.core.windows.net", STORAGE_KEY)

BASE_INPUT  = "wasbs://" + CONTAINER + "@" + STORAGE_ACCOUNT + ".blob.core.windows.net/raw_data"
BASE_BRONZE = "wasbs://" + CONTAINER + "@" + STORAGE_ACCOUNT + ".blob.core.windows.net/bronze"

In [0]:
"""
Pipeline Medallion - Camada BRONZE
Ingestao raw. Sem transformacao so traz pra dentro do lake e particiona por data de ingestao.
"""
from pyspark.sql import functions as F
from datetime import datetime

ingestion_date = datetime.now().strftime('%Y-%m-%d')

def ingest_to_bronze(source_file: str, table_name: str):
    """Le CSV, adiciona metadata de ingestao, salva como Delta."""
    print(f'>>> Ingerindo {source_file} -> bronze.{table_name}')
    
    df = (spark.read
          .option('header', True)
          .option('inferSchema', False)   # bronze mantem tudo string
          .csv(f'{BASE_INPUT}/{source_file}'))
    
    df_with_meta = (df
        .withColumn('_ingestion_timestamp', F.current_timestamp())
        .withColumn('_ingestion_date', F.lit(ingestion_date))
        .withColumn('_source_file', F.lit(source_file)))
    
    n = df_with_meta.count()
    print(f'    {n:,} linhas')
    
    (df_with_meta
        .write
        .mode('append')
        .partitionBy('_ingestion_date')
        .format('delta')
        .save(f'{BASE_BRONZE}/{table_name}'))

ingest_to_bronze('pedidos.csv',      'pedidos')
ingest_to_bronze('pedido_itens.csv', 'pedido_itens')
ingest_to_bronze('pagamentos.csv',   'pagamentos')
ingest_to_bronze('produtos.csv',     'produtos')
ingest_to_bronze('lojas.csv',        'lojas')
ingest_to_bronze('clientes.csv',     'clientes')

print('\n=== BRONZE OK ===')

>>> Ingerindo pedidos.csv -> bronze.pedidos
    4,946 linhas
>>> Ingerindo pedido_itens.csv -> bronze.pedido_itens
    12,118 linhas
>>> Ingerindo pagamentos.csv -> bronze.pagamentos
    4,623 linhas
>>> Ingerindo produtos.csv -> bronze.produtos
    15 linhas
>>> Ingerindo lojas.csv -> bronze.lojas
    12 linhas
>>> Ingerindo clientes.csv -> bronze.clientes
    2,000 linhas

=== BRONZE OK ===


In [0]:
# Quick check - le de volta
print('Amostra bronze.pedidos:')
df = spark.read.format('delta').load(f'{BASE_BRONZE}/pedidos')
df.show(3, truncate=False)
print(f'Total linhas: {df.count()}')

Amostra bronze.pedidos:
+-----------+----------+-------+-------------------+---------+--------+------------+-----+-----------+-----------------------+---------------+------------+
|pedido_id  |cliente_id|loja_id|criado_em          |status   |subtotal|taxa_entrega|total|canal      |_ingestion_timestamp   |_ingestion_date|_source_file|
+-----------+----------+-------+-------------------+---------+--------+------------+-----+-----------+-----------------------+---------------+------------+
|PED00000001|CLI00939  |LJ12   |2025-09-01T11:37:06|PAGO     |58.6    |10.61       |69.21|WEB        |2026-05-17 14:51:44.185|2026-05-17     |pedidos.csv |
|PED00000002|CLI00652  |LJ08   |2025-09-01T14:14:04|CANCELADO|21.05   |6.15        |27.2 |APP_ANDROID|2026-05-17 14:51:44.185|2026-05-17     |pedidos.csv |
|PED00000003|CLI00644  |LJ07   |2025-09-01T11:34:22|PENDENTE |21.88   |10.78       |32.66|WEB        |2026-05-17 14:51:44.185|2026-05-17     |pedidos.csv |
+-----------+----------+-------+--------